In [61]:
# Imports

import pandas as pd
import numpy as np
from pathlib import Path

In [62]:
# File declarations

FILES_BASE_DIR = "data/"

FILES_FLOW_DIR = FILES_BASE_DIR + "flow/"
FILES_FLOW = [
    "Krapina-Kupljenovo_2000-2024.csv",
    "Krapina-Kupljenovo_2025.csv",
    "Krapina-ZlatarBistrica_2000-2024.csv",
    "Krapinica-Zabok_2000-2024.csv",
    "Krapinica-Zabok_2025.csv"
]

FILES_PRECIPITATION_DIR = FILES_BASE_DIR + "precipitation/"
FILES_PRECIPITATION = [
    "Krapina_2000-2020.csv",
    "Krapina_2021-2025_unconfirmed.csv",
    "Puntijarka_2000_unconfirmed.csv",
    "Puntijarka_2000-2020.csv",
    "Puntijarka_2020-2025_unconfirmed.csv",
    "Varazdin_2000-2024.csv",
    "Varazdin_2024-2025_unconfirmed.csv",
    "Zagreb-Maksimir_2000-2025.csv"
]

FILES_TEMPERATURE_DIR = FILES_BASE_DIR + "temperature/"
FILES_TEMPERATURE = [
    "Krapina_2000-2020.csv",
    "Krapina_2021-2025_unconfirmed.csv",
    "Puntijarka_2000_unconfirmed.csv",
    "Puntijarka_2000-2020.csv",
    "Puntijarka_2020-2025_unconfirmed.csv",
    "Varazdin_2000-2024.csv",
    "Varazdin_2024-2025_unconfirmed.csv",
    "Zagreb-Maksimir_2000-2025.csv"
]

In [63]:
# Check if all declared files exist

checks = []

for base_dir, files in [
    (FILES_FLOW_DIR, FILES_FLOW),
    (FILES_PRECIPITATION_DIR, FILES_PRECIPITATION),
    (FILES_TEMPERATURE_DIR, FILES_TEMPERATURE),
]:
    for filename in files:
        path = Path(base_dir) / filename
        if not path.exists():
            checks.append(str(path))

if not checks:
    print("All declared files exist.")
else:
    missing_df = pd.DataFrame(checks, columns=["Missing declared files"])
    print(missing_df)
    raise FileNotFoundError("Some declared files are missing.")

All declared files exist.


In [71]:
def parse_flow_data(file_path):
    """
    Parses flow CSV format with metadata.

     Returns:
        {
            "metadata": dict,
            "data": pandas.DataFrame
        }
    """

    file_path = Path(file_path)
    with open(file_path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    metadata = {}
    data_start_idx = None
    for i, line in enumerate(lines):
        # Detect start of data
        if ":" not in line:
            data_start_idx = i
            break

        # Metadata lines
        key, value = line.split(":", 1)
        metadata[key.strip()] = value.strip()

    if data_start_idx is None:
        raise RuntimeError("Data section not found in file: " + str(file_path))

    # Data section
    df = pd.read_csv(
        file_path,
        sep=";",
        skiprows=data_start_idx,
        decimal=","
    )

    df.columns = [c.strip() for c in df.columns]

    df["datetime"] = pd.to_datetime(
        df["Date"] + " " + df["Time"],
        format="%d.%m.%Y. %H:%M:%S"
    )

    value_col = [c for c in df.columns if "Value" in c][0]
    df = df.rename(columns={value_col: "value"})

    df = df[["datetime", "value"]]

    return {
        "metadata": metadata,
        "data": df
    }


def parse_meteo_data(file_path):
    """
    Parses precipitation/temperature CSV formats with/without metadata/header.

    Returns:
    {
        "metadata": dict,
        "data": pandas.DataFrame
    }
    """

    file_path = Path(file_path)

    with open(file_path, "r", encoding="utf-8") as f:
        lines = [next(f).strip() for _ in range(3)]

    metadata = {}

    has_metadata = "DDMMYYYY" in lines[1]

    data_start_idx = 0

    if has_metadata:
        first = lines[0].split(";")
        second = lines[1].split(";")

        metadata = {
            "station_id": first[0],
            "station_name": first[1],
            "latitude": first[2],
            "longitude": first[3],
            "Hs": first[4],
            "Hb": first[5],
            "parameter": second[2].strip(),
        }

        # data starts after:
        # metadata line
        # header line
        # units line
        data_start_idx = 3

    # Data section
    df = pd.read_csv(
        file_path,
        sep=";",
        skiprows=data_start_idx,
        header=None,
        usecols=[0, 1, 2],
        names=["hour", "date", "value"],
        dtype=str
    )

    # Prepend zeros to ensure correct parsing and properly transform 1-24h to 0-23h
    df["date"] = df["date"].str.zfill(8)
    hour = df["hour"].astype(int)
    base_datetime = pd.to_datetime(
        df["date"],
        format="%d%m%Y",
        errors="coerce"
    )
    df["datetime"] = base_datetime + pd.to_timedelta(hour, unit="h")

    df["value"] = pd.to_numeric(
        df["value"].str.replace(",", ".", regex=False),
        errors="coerce",
    )

    df = df[["datetime", "value"]]

    return {
        "metadata": metadata,
        "data": df
    }

In [ ]:
# parsed = parse_flow_data(Path(FILES_FLOW_DIR) / FILES_FLOW[1])
# print(parsed["metadata"])
# parsed["data"].head()
# parsed["data"].info()

parsed = parse_meteo_data(Path(FILES_PRECIPITATION_DIR) / FILES_PRECIPITATION[1])
print(parsed["metadata"])
print(parsed["data"].head())
parsed["data"].info()

{'station_id': '234', 'station_name': 'Krapina', 'latitude': '46.137694 [N]', 'longitude': '15.888194 [E]', 'Hs': 'Hs=202 [m]', 'Hb': 'Hb=203.09 [m]', 'parameter': 'RR'}
<class 'pandas.DataFrame'>
RangeIndex: 43824 entries, 0 to 43823
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   datetime  43824 non-null  datetime64[us]
 1   value     6264 non-null   float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 684.9 KB
